In [119]:
# %% Minimal setup from class

import os, json, textwrap, re, time
import requests

API_KEY  = os.getenv("OPENAI_API_KEY", "cse476")
API_BASE = os.getenv("API_BASE", "http://10.4.58.53:41701/v1")  
MODEL    = os.getenv("MODEL_NAME", "bens_model")              

SYSTEM_PROMPT = "You are a helpful assistant. Reply with only the final answer—no explanation."
TEMPERATURE   = 0.45 #Must be a float

def call_model_chat_completions(prompt: str,
                                system: str = SYSTEM_PROMPT,
                                model: str = MODEL,
                                temperature: float = TEMPERATURE,
                                timeout: int = 60) -> dict:
    """
    Calls an OpenAI-style /v1/chat/completions endpoint and returns:
    { 'ok': bool, 'text': str or None, 'raw': dict or None, 'status': int, 'error': str or None, 'headers': dict }
    """
    url = f"{API_BASE}/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type":  "application/json",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": 1000,
    }

    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
        status = resp.status_code
        hdrs   = dict(resp.headers)
        if status == 200:
            data = resp.json()
            text = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            return {"ok": True, "text": text, "raw": data, "status": status, "error": None, "headers": hdrs}
        else:
            # try best-effort to surface error text
            err_text = None
            try:
                err_text = resp.json()
            except Exception:
                err_text = resp.text
            return {"ok": False, "text": None, "raw": None, "status": status, "error": str(err_text), "headers": hdrs}
    except requests.RequestException as e:
        return {"ok": False, "text": None, "raw": None, "status": -1, "error": str(e), "headers": {}}

def self_evaluate(question, prediction, expected_answer, model=MODEL):
    """
    Use the model itself as a strict grader.
    Returns True if the model says the prediction matches the expected answer; else False.
    Falls back to a simple normalized string compare if the model's reply is malformed.
    """
    import re

    system = "You are a strict grader. Reply with exactly True or False. No punctuation. No explanation."
    prompt = f"""You are grading a question-answer pair.

Return exactly True if the PREDICTION would be accepted as correct for the EXPECTED_ANSWER.
Otherwise, return False.

QUESTION:
{question}

PREDICTION:
{prediction}

EXPECTED_ANSWER:
{expected_answer}

Answer with exactly: True or False
"""

    r = call_model_chat_completions(
        prompt,
        system=system,
        model=model,
        temperature=0.0,
    )

    reply = (r.get("text") or "").strip().lower()
    if reply.startswith("true"):
        return True
    if reply.startswith("false"):
        return False

    # Fallback: simple normalization-based equality
    norm = lambda s: re.sub(r"\s+", " ", (s or "").strip().lower())
    return norm(prediction) == norm(expected_answer)

def self_evaluate_tests(tests, model=MODEL, grader_model=None, sleep_sec=0.2, verbose=True):
    """
    Run the tests by querying the model for each prompt, then use LLM-as-a-judge
    (self_evaluate) to determine correctness.

    Args:
        tests: list of dicts with keys: id, prompt, expected (and optionally type)
        model: model used to generate predictions
        grader_model: model used to judge correctness (defaults to `model` if None)
        sleep_sec: small delay between calls to be polite to the API
        verbose: if True, print a summary line per test

    Returns:
        rows: list of dicts with fields:
              id, expected, got, correct, status, error
    """
    import time

    judge_model = grader_model or model
    rows = []

    for t in tests:
        #1) Get model prediction
        if t.get("input"):
            r = call_model_chat_completions(
                t["input"],
                system="You are a careful solver. Reply ONLY with the final answer, nothing else.",
                model=model,
                temperature=TEMPERATURE,
            )
            got = agent_loop(t["input"])
            #got = reasoning_via_planning(t["input"])
            # 2) LLM-as-a-judge: strict True/False
            is_correct = self_evaluate(
                question=t["input"],
                prediction=got,
                expected_answer=t["output"],
                model=judge_model,
            )
        else:
            r = call_model_chat_completions(
                t["prompt"],
                system="You are a careful solver. Reply ONLY with the final answer, nothing else.",
                model=model,
                temperature=TEMPERATURE,
            )        #got = (r.get("text") or "").strip()
            got = agent_loop(t["prompt"])

            # 2) LLM-as-a-judge: strict True/False
            is_correct = self_evaluate(
                question=t["prompt"],
                prediction=got,
                expected_answer=t["expected"],
                model=judge_model,
            )



        row = {
            "id": t.get("id", "<unnamed>"),
            "output": t["output"],
            "got": got,
            "correct": bool(is_correct),
            "status": r.get("status"),
            "error": r.get("error"),
        }
        rows.append(row)

        if verbose:
            mark = "✅" if is_correct else "❌"
            print(f"{mark} {row['id']}: output={row['output']!r}, got={row['got']!r} (HTTP {row['status']})")
            if row["error"]:
                print("   error:", row["error"])

        if sleep_sec:
            time.sleep(sleep_sec)

    return rows


In [120]:
def reasoning_via_planning(prior:str, question: str,) -> dict:
    prior_reasoning = "\nPrior Reasoning: " + prior + "\n\n"
    reasoning_str = "Create a plan using as little words as possible. Using prior reasoning, decompose the problem into a step by step plan to solve the question provided. Once you have a plan, execute each step in order to arrive at the final answer. Make sure your answer solves the question provided.\n\n Question: "

    r = call_model_chat_completions(
            reasoning_str + question + prior_reasoning,
            system="You are a planner. Provide a structured step-by-step plan to solve the question.",
            model=MODEL,
            temperature=0.0,
        )
    got = (r.get("text") or "").strip()
    return got

In [121]:
def tree_of_thought(question: str, n_paths: int, prior: str = None,):
    tot_str = "You will decompose the problem down into {n_paths} distinct possible solution paths to solve the question provided. Depending on the problem, write a reasonable path that is different from every othe path created but still leads to the answer. Keep in mind you have limited word count to use, so be efficient and limit word output. Expected output should be in the form of: path1:<>, \npath2:<>,etc.\n\n Question: "
    r = call_model_chat_completions(
            tot_str.format(n_paths=n_paths) + question,
            system=SYSTEM_PROMPT,
            model=MODEL,
            temperature=TEMPERATURE,
        )
    #Divide into n_paths
    raw = r.get("text") or ""
    thoughts = []
    for n in range(1, n_paths + 1):
        path = raw.find(f"path{n}:")
        end = raw.find(f"path{n+1}:")
        if end == -1:
            end = len(raw)
        thoughts.append(raw[path:end].strip())
    return thoughts

In [ ]:
def double_check(prior:str, question: str):
    original_question = "Original Question: " + question + "\n"
    check_str = "You are a careful solver. Use the solution to answer the question. If you find any mistakes, correct them and provide the accurate final answer. If everything is correct, simply confirm the final answer.\n\n Solution to double-check: "
    r = call_model_chat_completions(
            original_question + check_str + prior,
            system="You are a intelligent grader/judge whose job is to validate whether solutions are correct. Given the question and solution, if correct: ouput the final answer. If incorrect: output incorrect, and provide the correct answer. If the answer is a numeric, provide the numeric answer only.",
            model=MODEL,
            temperature=TEMPERATURE,
        )
    got = (r.get("text") or "").strip()
    return got

In [ ]:
def critic(prior:str, question: str):
    original_question = "Original Question: " + question + "\n\n"
    crit_str = "Analyze the following solution summaries and choose the single best final answer. Do NOT include explanations — output only the final concise answer (as it should be given to the user).\n\Solutions:\n"
    r = call_model_chat_completions(
            original_question + crit_str + prior,
            system="You are a meticulous critic. Given multiple solution summaries, select the best final answer without any explanations. If the answer is numeric, provide the numeric answer only.",
            model=MODEL,
            temperature=TEMPERATURE,
        )
    got = (r.get("text") or "").strip()
    return got

<>:3: SyntaxWarning: invalid escape sequence '\S'
<>:3: SyntaxWarning: invalid escape sequence '\S'
C:\Users\isami\AppData\Local\Temp\ipykernel_20192\3897666830.py:3: SyntaxWarning: invalid escape sequence '\S'
  crit_str = "Analyze the following solution summaries and choose the single best final answer. Do NOT include explanations — output only the final concise answer (as it should be given to the user).\n\Solutions:\n"


In [124]:
import json
import random

with open("cse476_final_project_dev_data.json", "r") as tests:
    DEV_DATA = json.load(tests)

#Get test batches by domain/random
def filter_domain(domain: str):
    filtered = []
    for test in DEV_DATA:
        if test.get("domain") == domain:
            filtered.append(test)
    return filtered

def get_batch(num: int, domain: str = None, is_random: bool = False):
    random.seed(315)
    if domain:
        data = filter_domain(domain)
    else:
        data = DEV_DATA

    if is_random:
        return random.sample(data, num)
    else:
        return data[:num]
    

In [125]:
# Tree of thought (X of thought)
# Reasoning via planning
# 'Wait' am i correct? Double check
# Critic 
# Send to output

# Future:
# Implement RAG or memory of some kind to grab from text data
# implement In-context learning with examples via RAG
# Call to Wikipedia API/ disctionary API/ Calc?

def agent_loop(input_question: str):
    #First split into thoughts
    thoughts = tree_of_thought(input_question, n_paths=3)
    #Reason through each thought path
    for i, t in enumerate(thoughts):
        thoughts[i] = reasoning_via_planning(t, input_question)
        print(f"Thought: {thoughts[i]}")
    #Implement wait am i correct / double check
    for i, t in enumerate(thoughts):
        thoughts[i] = double_check(t, input_question)
        print(f"Double Checked Thought: {thoughts[i]}")
    #Evaluator / critic to pick best answer
    critic_str = "These are the current paths and their solutions: "
    for t in thoughts:
        critic_str += t + "\n"
    critic_str += "Given these solutions, pick the best one and provide the final answer only"
    best_answer = critic(critic_str, input_question)
    return best_answer

In [126]:

# Example:
#test = reasoning_via_planning("A farmer has 17 sheep and all but 9 are lost. How many sheep are left on the farm?")
#test2 = call_model_chat_completions("Make a plan to solve: A farmer has 17 sheep and all but 9 are lost. How many sheep are left on the farm?")
#print(test)
#print(test2)

tests = get_batch(1, domain="math", is_random=False)
self_evaluate_tests(tests, model=MODEL, sleep_sec=0.5, verbose=True)

Thought: 1. Use area condition → diagonals bisect each other  
2. Diagonals bisect each other → quadrilateral is a parallelogram  
3. In parallelogram, opposite sides are equal → AB = CD, BC = AD  
4. Given AB = CD = 10, BC = 14, AD = 2√65 → check if BC = AD  
5. BC = 14, AD = 2√65 ≈ 16.12 → not equal → contradiction  
6. Therefore, diagonals do not bisect each other  
7. Use area condition → diagonals are perpendicular  
8. If diagonals are perpendicular, area = ½ × AC × BD  
9. Use coordinates to find AC and BD  
10. Let A = (0, 0), B = (10, 0), D = (x, y), C = (x + 14, y)  
11. Use AD = 2√65 → x² + y² = (2√65)² = 4 × 65 = 260  
12. Use BC = 14 → (x + 14 - 10)² + (y - 0)² = 14² = 196  
13. Solve equations: x² + y² = 260 and (x + 4)² + y² = 196  
14. Expand: x² + 8x + 16 + y² = 196 → x² + y² + 8x = 180  
15. Substitute x² + y² = 260 → 260 + 8x = 180 → 8x = -80 → x = -10  
16. Substitute x = -10 into x² + y² = 260 → 100 + y² = 260 → y² = 160 → y = ±√160 = ±4√10  
17. Coordinates: A = (

[{'id': '<unnamed>',
  'output': '112',
  'got': '260',
  'correct': False,
  'status': 200,
  'error': None}]